In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline
from peft import PeftModel
import torch
import json
import time
import os
import random
from spacy_tester import extract_entities


# CONFIG

# Model A: User
BASE_LLAMA_PATH = "/home/hpc/v132ca/v132ca25/llm-dialog-project/llama3.2-3b-4bit"
LORA_ADAPTER_PATH = "/home/hpc/v132ca/v132ca25/llm-dialog-project/travel-lora"

# Model B: Assistant
MODEL_B_PATH = "/home/hpc/v132ca/v132ca25/llm-dialog-project/gemma-2b-travel-qa/snapshots/d718695bf69d2f1a630d34703c1d578ca068ba32"           

# Number of turns
MAX_TURNS = 10

# Load Ontology
with open("ontology_checklist.json", "r", encoding="utf-8") as f:
    ontology = json.load(f)
k = random.randint(1, len(ontology))
random_items = random.sample(list(ontology.keys()), k)
ontology = {key: ontology[key] for key in random_items}
print(f"Ontology{ontology}\n")


# Load Dictionary
with open("semantic_dictionary.json", "r", encoding="utf-8") as f:
    dictionary = json.load(f)


# Load Personas    
with open("personas.json", "r") as f:
    personas = json.load(f)
selected_persona = random.choice(personas)
print("User Persona:", selected_persona['name'])
    
    
# -------------------------
# LOAD MODEL A (LLaMA model + LoRA adapter)
# -------------------------

tokenizerA = AutoTokenizer.from_pretrained(BASE_LLAMA_PATH)

modelA_base = AutoModelForCausalLM.from_pretrained(
    BASE_LLAMA_PATH,
    dtype=torch.float16,
    device_map="auto"
)

modelA = PeftModel.from_pretrained(
    modelA_base,
    LORA_ADAPTER_PATH
)

pipelineA = pipeline(
    "text-generation",
    model=modelA,
    tokenizer=tokenizerA,
    max_new_tokens=100,
    temperature=0.7
)

# -------------------------
# LOAD MODEL B (Answer Model, chatbot)
# -------------------------

tokenizerB = AutoTokenizer.from_pretrained(MODEL_B_PATH)

modelB = AutoModelForCausalLM.from_pretrained(
    MODEL_B_PATH,
    dtype=torch.float16,
    device_map="auto"
)

pipelineB = pipeline(
    "text-generation",
    model=modelB,
    tokenizer=tokenizerB,
    max_new_tokens=200,
    temperature=0.5 #0.5 is strict in grammar and word choice, 0.7 is more natural
)


# -------------------------
# HELPER FUNCTION
# -------------------------


def chat(model_pipeline, prompt: str):
    """Generate a single response from a model."""
    output = model_pipeline(
        prompt,
        return_full_text=False  
    )[0]["generated_text"]
    return output.strip()


# -------------------------
# MAIN MULTI-TURN LOOP
# -------------------------

def run_dialogue(initial_persona_prompt: str):
    conversation = []
    current_turn = 1
    # ---- TURN 1 ----
    message_A = chat(pipelineA, initial_persona_prompt)
    print(f"TURN 1 >>> Model A: {message_A}\n")
    conversation.append(("Model A", message_A))
    
    if message_A:
        TRAVEL_GUIDE_PROMPT = (
            "You are an expert travel guide.\n"
            f"The User asked: {message_A}\n"
            "Give only ONE answer to the user's travel question, concisely and specifically.\n"
            "Focus only on practical information about the city and travel plan.\n"
            "Use a formal and professional tone.\n"
            "Do not ask follow-up questions.\n"
            "Keep your answer under 120 words.\nAnswer:"
        )
    
    message_B = chat(pipelineB, TRAVEL_GUIDE_PROMPT)
    print(f"TURN 1 >>> Model B: {message_B}\n")
    conversation.append(("Model B", message_B))
    
    current_key = None
    
    # ---- NEXT TURNS ----
    for turn in range(2, MAX_TURNS + 1):
        # Check for unpopulated ontology keys
        null_keys = [key for key, value in ontology.items() if value["populated"] is None]
        if not null_keys:
            break
        
        # Select key for this turn
        if current_key is None:
            selected_key = random.choice(null_keys)
            current_key = selected_key
        else:
            selected_key = current_key
        
        print(f"\n{'='*50}")
        current_turn = turn
        print(f"TURN {turn} - Processing: {selected_key}")
        print(f"Ontology status: {ontology}")
        print(f"{'='*50}\n")
        
        # ----- USER QUESTION -----
        next_promptA = (
            "You are a user-simulator.\n"
            f"Previous conversation: {conversation[-4:] if len(conversation) > 4 else conversation}\n"
            f"Ask ONLY ONE specific travel-related question about {selected_key}.\n"
            "Make it a single, short, clear question.\n"
            "Do NOT repeat previous questions.\n"
            "Question:"
        )
        
        message_A = chat(pipelineA, next_promptA)
        print(f"USER: {message_A}")
        conversation.append(("Model A", message_A))
        
        # ----- TRAVEL GUIDE ANSWER -----
        next_promptB = (
            "You are an expert travel guide.\n"
            f"Previous conversation context: {conversation[-4:] if len(conversation) > 4 else conversation}\n"
            "Answer the user's question clearly and concisely.\n"
            "Provide practical, specific information.\n"
            "Keep your answer under 100 words.\n"
            f"User question: {message_A}\n"
            "Answer:"
        )
        
        message_B = chat(pipelineB, next_promptB)
        print(f"GUIDE: {message_B}")
        conversation.append(("Model B", message_B))
        
        # ----- ONTOLOGY POPULATION LOGIC -----
        try:
            # Extract entities using Spacy
            entities = extract_entities(message_B, selected_key)
            
            if entities:
                # Check if we have too many entities (need follow-up)
                if len(entities) > 3:
                    print(f"Too many entities found ({len(entities)}). Initiating follow-up question....")
                    
                    # Store partial entities
                    ontology[selected_key]["entities"] = entities
                    ontology[selected_key]["populated"] = False
                    
                    # ----- FOLLOW-UP QUESTION -----
                    followup_promptA = (
                        "You are a user-simulator.\n"
                        f"The travel guide gave too many options for {selected_key}: {entities}\n"
                        "Ask them to narrow it down to just ONE specific recommendation.\n"
                        "Make it a single polite request under 30 words.\n"
                        "Request:"
                    )
                    
                    followup_A = chat(pipelineA, followup_promptA)
                    print(f"FOLLOW-UP USER: {followup_A}")
                    conversation.append(("Model A", followup_A))
                    
                    # Follow-up answer
                    followup_promptB = (
                        "You are an expert travel guide.\n"
                        f"The user wants you to narrow down the options for {selected_key}.\n"
                        "Choose ONE specific, best recommendation from your previous answer.\n"
                        "Explain briefly why you chose it.\n"
                        f"Previous options: {entities}\n"
                        "Recommendation:"
                    )
                    
                    followup_B = chat(pipelineB, followup_promptB)
                    print(f"FOLLOW-UP GUIDE: {followup_B}")
                    conversation.append(("Model B", followup_B))
                    
                    # Extract entities from follow-up
                    followup_entities = extract_entities(followup_B, selected_key)
                    if followup_entities:
                        # Take only the first entity from follow-up
                        ontology[selected_key]["entities"] = followup_entities[:1]
                        ontology[selected_key]["populated"] = True
                        current_key = None  # Move to next category
                        print(f"✓ {selected_key} populated with: {followup_entities[:1]}")
                    else:
                        # If still no entities, keep trying with same category
                        print(f"✗ No entities in follow-up. Keeping {selected_key} unpopulated.")
                        current_key = selected_key
                
                else:
                    # 1-3 entities is good
                    ontology[selected_key]["entities"] = entities
                    ontology[selected_key]["populated"] = True
                    current_key = None  # Move to next category
                    print(f"✓ {selected_key} populated with: {entities}")
            
            else:
                # No entities found
                print(f"✗ No entities found for {selected_key}.")
                ontology[selected_key]["populated"] = False
                current_key = selected_key  # Try same category again
        
        except Exception as e:
            print(f"Error in ontology population: {e}")
            current_key = selected_key  # Try same category again
        
        # Print current ontology state
        print(f"\nCurrent ontology: {ontology}")
        print(f"{'='*50}\n")
    
    # ---- FINAL OUTPUT ----

    
    happy_final_promptA = (f"You are a {selected_persona['age']} {selected_persona['name']}.\n"
                           f"Previous conversation: {conversation[-4:] if len(conversation) > 4 else conversation}"
    "Task: Write ONE informal sentence (under 30 words) expressing satisfaction with the travel planning conversation.\n"
    "Rules:\n"
    "- Output ONLY the sentence.\n"
    "- Do NOT explain the task.\n"
    "- Do NOT continue the dialogue.\n"
    "Sentence:")
    
    dissatisfied_final_promptA = (
    f"You are a {selected_persona['age']} {selected_persona['name']}.\n"
    f"Previous conversation: {conversation[-4:] if len(conversation) > 4 else conversation}"
    "Task: Write ONE informal, annoyed sentence (under 30 words) expressing dissatisfaction.\n"
    "Rules:\n"
    "- Use simple, natural language.\n"
    "- Output ONLY the sentence.\n"
    "- Do NOT explain the task.\n"
    "- Do NOT continue the dialogue.\n"
    "Sentence:")
    
    if current_turn > 8:
        message_A = chat(pipelineA, dissatisfied_final_promptA)
    else:
        message_A = chat(pipelineA, happy_final_promptA)
    print(f"USER: {message_A}")
    conversation.append(("Model A", message_A))
    
    final_promptB = (
    "Respond to the traveler's message with ONE polite acknowledgment sentence."
    f"User said: {message_A}\n"
    "Your response:"
    )
    message_B = chat(pipelineB, final_promptB)
    print(f"GUIDE: {message_B}")
    conversation.append(("Model B", message_B))

    
    print("\n" + "="*60)
    print("CONVERSATION COMPLETE")
    print("="*60)
    print(f"Final ontology state:")
    for key, value in ontology.items():
        populated = "✓" if value["populated"] else "✗" if value["populated"] is False else "?"
        print(f"  {populated} {key}: {value['entities']}")
    print("="*60)

    os.makedirs("conversation_logs", exist_ok=True)
    filename = f"conversation_logs/conv_{int(time.time())}.json"

    # Save the conversation
    with open(filename, "w", encoding="utf-8") as f:
        json.dump({"persona": selected_persona['name'], "conversation": conversation, 
                   "ontology": ontology, "timestamp": time.time()}, f, indent=2)

    print(f"✓ Conversation saved to: {filename}")

    return conversation, ontology

# -------------------------
# RUN
# -------------------------

if __name__ == "__main__":
    print("Running multi-turn conversation between two local LLMs...\n")
            
    initial_prompt = (
    f"You are a {selected_persona['age']} {selected_persona['name']}. "
    "Generate ONE realistic travel question, about a general first-impression question about ONE specific city in Europe"
    "(opinion and vibes, or what it's known for). "
    f"You prefer {', '.join(selected_persona['preferences'])}. "
    "Use a curious, informal tone, max 30 words."
    )

    conversation = run_dialogue(initial_prompt)
 
        
# =========================
# CLEANUP
# =========================
del pipelineA
del pipelineB
del modelA
del modelA_base
del modelB
del tokenizerA
del tokenizerB

torch.cuda.empty_cache()

/apps/jupyterhub/jh3.1.1-py3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeyboardInterrupt: 